# Lab 2: Training & Transfer

Training quality depends on choices that interact: optimizer step size, batch
size, schedule, initialization, and data regime. This lab gives you a complete
compact-CNN training loop and a controlled transfer-learning comparison. You
will tune the loop, then compare an ImageNet-pretrained ResNet-18 with the same
architecture trained from random initialization on a small CIFAR-10 subset.

No implementation tasks remain. Your job is to run controlled comparisons,
diagnose curve behavior, and justify a practitioner decision.

**Learning objectives**

- Connect dataset size and batch size to optimizer steps per epoch.
- Diagnose an overly aggressive learning rate from the training curve.
- Compare batch and schedule changes without silently changing other factors.
- Explain why pretrained visual features can help in a small-data regime.


## How this notebook is assessed

The edX submission grades one deterministic loader calculation and two
scenario-based decisions. It does not grade the accuracy produced by your
Colab session. The transfer comparison uses a fixed 5,000-example target
training set, so the loader diagnostic remains the same in quick and full
modes.

Quick mode shortens the compact-CNN sweeps and uses 1,000 validation images.
Full mode uses 5,000 validation images and longer compact-CNN sweeps. The
transfer comparison uses five epochs in either mode, and the official
CIFAR-10 test split remains untouched.


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import asdict, dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")
if DEVICE.type == "cpu":
    print("A free Colab GPU is recommended for the ResNet-18 comparison.")


In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

# The 5,000-example training subset is invariant because the graded loader
# diagnostic and transfer comparison both depend on it.
TARGET_TRAIN_PER_CLASS = 500
TARGET_VALIDATION_PER_CLASS = 100 if QUICK_MODE else 500
DEFAULT_BATCH_SIZE = 128
TUNING_EPOCHS = 3 if QUICK_MODE else 6
TRANSFER_EPOCHS = 5
TRANSFER_IMAGE_SIZE = 96
NUM_WORKERS = 0 if QUICK_MODE else 2

print(
    f"quick_mode={QUICK_MODE}, target train={10 * TARGET_TRAIN_PER_CLASS:,}, "
    f"validation={10 * TARGET_VALIDATION_PER_CLASS:,}, "
    f"transfer epochs={TRANSFER_EPOCHS}"
)


## 1. Build one fixed target split with several transform views

We use the same stratified indices for every compact-CNN and ResNet-18 run.
The compact CNN receives 32×32 inputs with CIFAR normalization. ResNet-18
receives 96×96 inputs with ImageNet normalization. Only training views use
random crops or flips; validation views are deterministic.

The source notebooks provide CIFAR loaders, CNN construction, and training-loop
patterns, but they do not include this transfer workflow. ResNet-18 plus the
fixed CIFAR-10 subset is an explicit online-course adaptation.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

tiny_train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=3, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)
tiny_eval_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
)
transfer_train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(
            TRANSFER_IMAGE_SIZE, scale=(0.75, 1.0)
        ),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
transfer_eval_transform = transforms.Compose(
    [
        transforms.Resize((TRANSFER_IMAGE_SIZE, TRANSFER_IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(
    targets: list[int], train_per_class: int, validation_per_class: int, seed: int
) -> tuple[list[int], list[int]]:
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices: list[int] = []
    validation_indices: list[int] = []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        if needed > len(class_indices):
            raise ValueError(f"Requested {needed} examples from class {class_id}")
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


target_train_indices, target_validation_indices = stratified_split(
    split_source.targets,
    TARGET_TRAIN_PER_CLASS,
    TARGET_VALIDATION_PER_CLASS,
    SEED,
)
assert set(target_train_indices).isdisjoint(target_validation_indices)

tiny_train_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=tiny_train_transform, download=False
)
tiny_eval_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=tiny_eval_transform, download=False
)
transfer_train_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=transfer_train_transform, download=False
)
transfer_eval_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=transfer_eval_transform, download=False
)

tiny_train_set = Subset(tiny_train_source, target_train_indices)
tiny_train_eval_set = Subset(tiny_eval_source, target_train_indices)
tiny_validation_set = Subset(tiny_eval_source, target_validation_indices)
transfer_train_set = Subset(transfer_train_source, target_train_indices)
transfer_train_eval_set = Subset(
    transfer_eval_source, target_train_indices
)
transfer_validation_set = Subset(
    transfer_eval_source, target_validation_indices
)


def seeded_loader(
    dataset,
    *,
    batch_size: int,
    shuffle: bool,
    seed: int = SEED,
) -> DataLoader:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


diagnostic_loader = seeded_loader(
    tiny_train_set,
    batch_size=DEFAULT_BATCH_SIZE,
    shuffle=True,
)
optimizer_steps_per_epoch = len(diagnostic_loader)
assert len(tiny_train_set) == 5_000
assert optimizer_steps_per_epoch == 40
print("target-training examples:", len(tiny_train_set))
print("validation examples:", len(tiny_validation_set))
print(
    "Deterministic diagnostic — optimizer steps per epoch:",
    optimizer_steps_per_epoch,
)
print("official CIFAR-10 test split: reserved and not loaded")


**Hand-check the loader diagnostic.** Thirty-nine full batches contain
(39(128)=4{,}992) examples. Because `drop_last=False`, the remaining eight
examples make one final batch. Enter the printed number of optimizer steps in
the first Lab 2 edX problem.


## 2. Use the provided compact CNN and training loop

The compact CNN makes optimizer sweeps inexpensive. The helper evaluates
training accuracy on the deterministic, non-augmented view so its gap from
validation accuracy is interpretable.


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images).flatten(1)
        return self.classifier(features)


@torch.no_grad()
def evaluate_classifier(
    model: nn.Module, loader: DataLoader
) -> tuple[float, float]:
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.numel()
    return total_loss / total_examples, total_correct / total_examples


def fit_classifier(
    *,
    label: str,
    model: nn.Module,
    train_set,
    train_eval_set,
    validation_set,
    batch_size: int,
    epochs: int,
    learning_rate: float,
    weight_decay: float = 1e-4,
    schedule: str = "constant",
    seed: int = SEED,
    verbose: bool = True,
) -> dict:
    if schedule not in {"constant", "step"}:
        raise ValueError("schedule must be 'constant' or 'step'")
    fit_loader = seeded_loader(
        train_set, batch_size=batch_size, shuffle=True, seed=seed
    )
    train_eval_loader = seeded_loader(
        train_eval_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=batch_size, shuffle=False, seed=seed
    )
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    scheduler = None
    if schedule == "step":
        scheduler = torch.optim.lr_scheduler.StepLR(
            optimizer, step_size=max(1, epochs // 2), gamma=0.3
        )
    loss_fn = nn.CrossEntropyLoss()
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "validation_loss": [],
        "validation_accuracy": [],
        "learning_rate": [],
    }
    started = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()

        train_loss, train_accuracy = evaluate_classifier(model, train_eval_loader)
        validation_loss, validation_accuracy = evaluate_classifier(
            model, validation_loader
        )
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        history["learning_rate"].append(optimizer.param_groups[0]["lr"])
        if verbose:
            print(
                f"{label:>18} | epoch {epoch:>2}/{epochs} | "
                f"lr {optimizer.param_groups[0]['lr']:.1e} | "
                f"train acc {train_accuracy:6.2%} | "
                f"val acc {validation_accuracy:6.2%}"
            )
        if scheduler is not None:
            scheduler.step()

    record = {
        "label": label,
        "batch_size": batch_size,
        "epochs": epochs,
        "initial_learning_rate": learning_rate,
        "schedule": schedule,
        "steps_per_epoch": len(fit_loader),
        "seconds": time.perf_counter() - started,
        "history": history,
    }
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


def plot_history(record: dict) -> None:
    history = record["history"]
    epochs = np.arange(1, len(history["train_accuracy"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
    axes[0].plot(
        epochs, history["validation_loss"], marker="o", label="validation"
    )
    axes[0].set(title=f"{record['label']}: loss", xlabel="epoch", ylabel="loss")
    axes[0].legend()
    axes[1].plot(
        epochs, history["train_accuracy"], marker="o", label="train"
    )
    axes[1].plot(
        epochs,
        history["validation_accuracy"],
        marker="o",
        label="validation",
    )
    axes[1].set(
        title=f"{record['label']}: accuracy",
        xlabel="epoch",
        ylabel="accuracy",
        ylim=(0, 1),
    )
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def experiment_summary(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        history = record["history"]
        rows.append(
            {
                "run": record["label"],
                "batch": record["batch_size"],
                "steps/epoch": record["steps_per_epoch"],
                "schedule": record["schedule"],
                "final lr": history["learning_rate"][-1],
                "final train acc": history["train_accuracy"][-1],
                "final validation acc": history["validation_accuracy"][-1],
                "seconds": record["seconds"],
            }
        )
    return pd.DataFrame(rows).set_index("run")


## 3. Sweep learning rate

These runs change only AdamW's initial learning rate. Inspect the loss curve,
not just final accuracy. Large oscillations or failure to settle are evidence
that updates may be too aggressive; a slowly descending curve can indicate a
conservative rate or simply too few epochs.


In [ ]:
learning_rate_results = []
for learning_rate in (3e-4, 3e-3, 3e-2):
    reset_seed()
    label = f"lr={learning_rate:g}"
    result = fit_classifier(
        label=label,
        model=TinyCNN(),
        train_set=tiny_train_set,
        train_eval_set=tiny_train_eval_set,
        validation_set=tiny_validation_set,
        batch_size=DEFAULT_BATCH_SIZE,
        epochs=TUNING_EPOCHS,
        learning_rate=learning_rate,
        schedule="constant",
    )
    learning_rate_results.append(result)

experiment_summary(learning_rate_results)


## 4. Compare batch size and schedule

Use 0.003 as a provided stable starting point, then compare one change at a
time. Notice that batch size changes the number of optimizer steps in an
epoch: compare both epochs and `steps/epoch` rather than assuming an epoch is
the same optimization budget. The schedule comparison keeps batch size fixed.


In [ ]:
optimizer_experiments = [
    ("batch 128, constant", 128, "constant"),
    ("batch 64, constant", 64, "constant"),
    ("batch 128, step", 128, "step"),
]
optimizer_results = []
for label, batch_size, schedule in optimizer_experiments:
    reset_seed()
    optimizer_results.append(
        fit_classifier(
            label=label,
            model=TinyCNN(),
            train_set=tiny_train_set,
            train_eval_set=tiny_train_eval_set,
            validation_set=tiny_validation_set,
            batch_size=batch_size,
            epochs=TUNING_EPOCHS,
            learning_rate=3e-3,
            schedule=schedule,
        )
    )

experiment_summary(optimizer_results)


## 5. Compare pretrained and random initialization

Both runs below use ResNet-18, replace its 1,000-class ImageNet head with a new
ten-class head, and update **all** layers. They use the same target split,
transforms, optimizer, seed, and five-epoch budget. The only intended
difference is whether the backbone begins with ImageNet-pretrained weights.

The pretrained weights are about a 45 MB one-time download. At 96×96 and only
5,000 training examples, both runs are designed for a free Colab GPU. They are
runnable on CPU but will take longer.


In [ ]:
def make_resnet18(*, pretrained: bool) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    input_features = model.fc.in_features
    if input_features != 512:
        raise RuntimeError(f"Expected a 512-feature ResNet-18 head, got {input_features}")
    model.fc = nn.Linear(input_features, 10)
    return model


def run_transfer_comparison(*, pretrained: bool) -> dict:
    reset_seed()
    label = "ImageNet pretrained" if pretrained else "random initialization"
    model = make_resnet18(pretrained=pretrained)
    replacement_head_parameters = sum(
        parameter.numel() for parameter in model.fc.parameters()
    )
    assert replacement_head_parameters == 5_130
    print(f"{label}: replacement-head parameters = {replacement_head_parameters:,}")
    return fit_classifier(
        label=label,
        model=model,
        train_set=transfer_train_set,
        train_eval_set=transfer_train_eval_set,
        validation_set=transfer_validation_set,
        batch_size=DEFAULT_BATCH_SIZE,
        epochs=TRANSFER_EPOCHS,
        learning_rate=3e-4,
        weight_decay=1e-4,
        schedule="constant",
    )


transfer_results = [
    run_transfer_comparison(pretrained=True),
    run_transfer_comparison(pretrained=False),
]
experiment_summary(transfer_results)


## Reflection and report

Use your real curves and tables to write a short private lab note:

1. Which learning rate was stable in your runtime? What curve evidence supports
   that claim?
2. How did changing batch size change steps per epoch? Why does that complicate
   comparisons at a fixed number of epochs?
3. What did the schedule change do to the loss trajectory?
4. Under the shared five-epoch budget, which initialization reached useful
   validation performance sooner? Explain the result in terms of reusable
   features, not a guaranteed accuracy number.

Measured values are not exact graded answers. On edX, submit the deterministic
loader count and answer the learning-rate and transfer scenarios.


## Submission checklist

- [ ] I entered R1, R2, and R3 from the final Report Values cell.
- [ ] I answered the unstable-learning-rate decision question.
- [ ] I answered the pretrained-versus-random-initialization question.
- [ ] I did not upload code or submit a run-specific accuracy as an exact value.

**Provenance:** The CIFAR loading and training-loop patterns are adapted from
`6_7960_Fall_2024_hw1_CIFAR10.ipynb` and Problem 1 of
`6_7960_Fall_2025_hw2_Transformers.ipynb`. Those notebooks contain no
pretrained/fine-tuning implementation; the fixed CIFAR-10 target subset and
ResNet-18 comparison are new online-course adaptations required by the lab
restructure specification.


## Report values

Run the cell below after completing the required experiments. It prints three
fixed, mode-independent integers that verify critical stages of the notebook: the
target training and validation splits are disjoint, the fixed 5,000-example loader
has the specified number of optimizer steps, and every required learning-rate and
transfer run reached its configured epoch count. Enter R1, R2, and R3 in the first
Lab 2 submission problem in MIT Learn.

Measured accuracy, loss, and timing are deliberately absent because they vary
across runtimes and are evidence for interpretation rather than exact answers.


In [ ]:
reported_histories = [*learning_rate_results, *transfer_results]
split_overlap_count = len(
    set(target_train_indices) & set(target_validation_indices)
)
complete_history_count = sum(
    len(record["history"]["train_accuracy"]) == record["epochs"]
    for record in reported_histories
)

report_values = {
    "R1 — overlapping target train/validation indices": split_overlap_count,
    "R2 — optimizer steps per epoch (fixed 5,000-example loader)": optimizer_steps_per_epoch,
    "R3 — complete learning-rate and transfer histories": complete_history_count,
}
assert tuple(report_values.values()) == (0, 40, 5)

print("LAB 2 REPORT VALUES (enter integers without separators)")
for label, value in report_values.items():
    print(f"{label}: {value}")
